In [1]:
!pip install skorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 271.6/271.6 kB 6.1 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader
from skorch import NeuralNetClassifier
from sklearn.model_selection import RandomizedSearchCV
import torch.nn as nn
import re
from sklearn.metrics import accuracy_score
from scipy.stats import randint, uniform
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.preprocessing import LabelEncoder
from datasets import Dataset


In [3]:
column_names = ["tweet id", "Entity", "Intent", "Post"]
df = pd.read_csv("/kaggle/input/datasets/shauryasayshi/twitter-x-dataset/twitter_training.csv", header=None, names=column_names)
test_df = pd.read_csv("/kaggle/input/datasets/shauryasayshi/twitter-x-dataset/twitter_validation.csv", header=None, names=column_names)
print(df.duplicated().sum())
print(df.isna().sum())
print("\nchecking placeholders..")
print(df.isin(['N/A', 'NA', '-999', '?']).sum())
df = df.dropna(subset=['Post']).reset_index(drop=True)
# df.columns

2700
tweet id      0
Entity        0
Intent        0
Post        686
dtype: int64

checking placeholders..
tweet id    0
Entity      0
Intent      0
Post        0
dtype: int64


In [4]:
train_df = df.dropna(subset=["Post", "Intent"])
val_df = test_df.dropna(subset=["Post", "Intent"])

label_encoder = LabelEncoder()

y_train = label_encoder.fit_transform(train_df["Intent"])
y_val = label_encoder.transform(val_df["Intent"])
num_classes2 = len(label_encoder.classes_)


tokenizer2 = AutoTokenizer.from_pretrained("distilbert-base-uncased")
train_tokens = tokenizer2(train_df['Post'].to_list(), padding=True, truncation=True, max_length = 35, return_tensors="pt")
test_tokens = tokenizer2(val_df['Post'].to_list(), padding=True, truncation=True, max_length = 35, return_tensors="pt")

# train_tensors = {
#     key : torch.tensor(val, dtype=torch.long)
#     for key, val in train_tokens.items()
# }
train_data = {
    'input_ids':train_tokens['input_ids'],
    'attention_mask': train_tokens['attention_mask'],
    'labels':y_train
}

val_data = {
    'input_ids': test_tokens['input_ids'],
    'attention_mask': test_tokens['attention_mask'],
    'labels':y_val
}

train_dataset = Dataset.from_dict(train_data).with_format("torch")
val_dataset = Dataset.from_dict(val_data).with_format("torch")
# train_tensors.keys()
# train_tensors['input_ids']
# train_dataset = TensorDataset(train_tensors["input_ids"], train_tensors["attention_mask"], torch.tensor(y_train, dtype=torch.long))
# val_dataset = TensorDataset(test_tokens["input_ids"], test_tokens["attention_mask"], torch.tensor(y_val, dtype=torch.long))

# test_tensors = torch.tensor(test_tokens, dtype=torch.long)
# train_tokens.keys()
# train_dataset = TensorDataset(train_tokens["input_"])
# train_tensors.keys()
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(train_dataset, batch_size=32)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = (preds == labels).mean()
    return {"accuracy": acc}

model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=num_classes2)

training_args = TrainingArguments(
    output_dir = "./results",
    num_train_epochs = 3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate = 2e-5,
    eval_strategy = "epoch",
    save_strategy = "epoch",
    logging_steps = 10,
    load_best_model_at_end = True,
    metric_for_best_model = "accuracy",
    report_to = "none"

)

trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset,
    compute_metrics = compute_metrics
)

trainer.train()
eval_results = trainer.evaluate()
print(eval_results)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 0.4935210645198822, 'eval_accuracy': 0.929, 'eval_runtime': 0.8524, 'eval_samples_per_second': 1173.147, 'eval_steps_per_second': 18.77, 'epoch': 3.0}


In [5]:
X = df["Post"]
y = df["Intent"]
# X_train, X_test, y_train, y_test = train_test_splito(X, y, test_size=0.2, random_state=42)
test_X = test_df['Post']
test_y = test_df['Intent']

In [6]:
text = X.tolist()
test_text = test_X.tolist()
tokenizer2 = AutoTokenizer.from_pretrained("distilbert-base-uncased")
def tokenize(text):
    text = str(text).lower()
    return re.findall(r"\b\w+\b", text)
    return tokenizer(text, truncation=True, padding=True, return_tensor="pt")

tokens = [tokenize(sentence) for sentence in text]
test_tokens = [tokenize(sentence) for sentence in test_text]
# test_tokens

In [7]:
vocab = {
    "<PAD>":0,
    "<UNK>":1
}

for token_list in tokens:
    for word in token_list:
        if word not in vocab:
            vocab[word] = len(vocab)

def encode(token_list):
    return [vocab.get(word, vocab["<UNK>"]) for word in token_list]
encoded_text = [encode(token_list) for token_list in tokens]
encoded_test_text = [encode(token_list) for token_list in test_tokens]

In [8]:
# max_length = max(len(x) for x in encoded_text)
max_length = 35
max_length

padded_text = []
padded_test_text = []

for token_list in encoded_text:
    if len(token_list) >= max_length:
        token_list = token_list[:max_length]
    else:
        padded_length = max_length - len(token_list)
        token_list += [0] * (padded_length)
    padded_text.append(token_list)

for token_list in encoded_test_text:
    if len(token_list) >= max_length:
        token_list = token_list[:max_length]
    else:
        padded_length = max_length - len(token_list)
        token_list += [0] * (padded_length)
    padded_test_text.append(token_list)


In [9]:
labels = sorted(y.unique())

labels_to_id = {
    label:i for i, label in enumerate(labels)
}

id_to_labels = {
    i:label for i, label in labels_to_id.items()
}

encoded_labels = [labels_to_id[label] for label in y]
encoded_test_labels = [labels_to_id[label] for label in test_y]

In [10]:
X_train_tensor = torch.tensor(padded_text, dtype=torch.long)
X_test_tensor = torch.tensor(padded_test_text, dtype=torch.long)
y_train_tensor = torch.tensor(encoded_labels, dtype=torch.long)
y_test_tensor = torch.tensor(encoded_test_labels, dtype=torch.long)

In [11]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
train_loader = DataLoader(train_dataset, batch_size = 64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size = 64)

In [12]:
class Classifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, max_length, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.network = nn.Sequential(nn.Linear(embedding_dim, 128), nn.ReLU(), nn.Linear(128, num_classes))
    def forward(self, x):
        x = self.embedding(x)
        x = x.mean(dim=1)
        return self.network(x)


In [13]:
vocab_size = len(vocab)
num_classes = len(labels_to_id)
net = NeuralNetClassifier(Classifier, module__vocab_size=vocab_size,
                          module__max_length=max_length, 
                          module__num_classes=num_classes, 
                          max_epochs=100, 
                          criterion=nn.CrossEntropyLoss, 
                          optimizer = torch.optim.Adam,
                          device='cuda',
                          batch_size=256,
                          verbose=0)
net

<class 'skorch.classifier.NeuralNetClassifier'>[uninitialized](
  module=<class '__main__.Classifier'>,
  module__max_length=35,
  module__num_classes=4,
  module__vocab_size=31117,
)

In [14]:
params_grid={
    'module__embedding_dim': randint(32, 128),
    'lr':uniform(0.001, 0.01),
    'optimizer__weight_decay' : uniform(1e-6, 1e-3)
}

search = RandomizedSearchCV(estimator=net, param_distributions=params_grid, n_iter=5, cv=3, scoring="accuracy", error_score='raise')
search.fit(X_train_tensor, y_train_tensor)
# search.best_params_
# search.best_score_
y_pred = search.predict(X_test_tensor)
acc = accuracy_score(y_test_tensor, y_pred)
print(f"Proper Test Accuracy: {acc * 100:.2f}%\n")

Proper Test Accuracy: 77.10%

